In [1]:
import requests
import pandas as pd
from datetime import datetime
from pathlib import Path

# configuration globale (clé API et dossier de cache)
API_KEY = "101af1cc-e294-46b0-8ebc-9e9488b302f3"
CACHE_DIR = Path("cache_sirene")
CACHE_DIR.mkdir(exist_ok=True)

# extraction des établissements actifs (état A) d'un code postal donné
def extraire_secteur(cp: str, force_refresh: bool = False) -> pd.DataFrame:

    cache_file = CACHE_DIR / f"cp_{cp}_actifs.csv"

    if cache_file.exists() and not force_refresh:
        print(f"Cache trouvé pour {cp} — chargement local ({cache_file})")
        df = pd.read_csv(cache_file, dtype={"siret": str, "siren": str, "code_postal": str})
        df["date_creation"]  = pd.to_datetime(df["date_creation"], errors="coerce")
        df["annee_creation"] = df["annee_creation"].astype("Int64")
        return df
    
    url = "https://api.insee.fr/api-sirene/3.11/siret"
    headers_http = {
        "X-INSEE-Api-Key-Integration": API_KEY,
        "Accept": "application/json"
    }

    resultats = []
    curseur = "*"

    print(f"Extraction du CP {cp} depuis l'API SIRENE (filtre actifs appliqué localement)...")

    while True:
        params = {
            "q":       f"codePostalEtablissement:{cp}",
            "nombre":  1000,
            "curseur": curseur,
        }
        response = requests.get(url, headers=headers_http, params=params)

        if response.status_code != 200:
            print(f"\nErreur {response.status_code} : {response.text}")
            break

        data  = response.json()
        etabs = data.get("etablissements", [])

        if not etabs:
            print("\nFin des résultats.")
            break

        for e in etabs:
            periodes = e.get("periodesEtablissement", [{}])
            periode  = periodes[0] if periodes else {}

            etat = periode.get("etatAdministratifEtablissement", "")
            if etat != "A":   # ← on ignore les établissements fermés
                continue

            ape   = periode.get("activitePrincipaleEtablissement") or ""
            unite = e.get("uniteLegale", {})
            nom   = (
                unite.get("denominationUniteLegale")
                or f"{unite.get('prenomUsuelUniteLegale', '')} {unite.get('nomUniteLegale', '')}"
            )
            adresse = e.get("adresseEtablissement", {})

            date_creation_raw = e.get("dateCreationEtablissement")
            try:
                date_creation  = datetime.strptime(date_creation_raw, "%Y-%m-%d").date() if date_creation_raw else None
                annee_creation = date_creation.year if date_creation else None
            except ValueError:
                date_creation  = None
                annee_creation = None

            resultats.append({
                "nom":              nom.strip(),
                "siret":            e.get("siret", "N/A"),
                "siren":            e.get("siren", "N/A"),
                "ape":              ape,
                "code_postal":      adresse.get("codePostalEtablissement", "N/A"),
                "ville":            adresse.get("libelleCommuneEtablissement", "N/A"),
                "date_creation":    date_creation,
                "annee_creation":   annee_creation,
                "tranche_effectif": e.get("trancheEffectifsEtablissement"),
            })

        curseur_suivant = data.get("header", {}).get("curseurSuivant")
        print(f"  … {len(resultats)} actifs conservés jusqu'ici", end="\r")

        if not curseur_suivant or curseur_suivant == curseur:
            print("\nFin de la pagination.")
            break
        curseur = curseur_suivant

    df = pd.DataFrame(resultats)
    if not df.empty:
        df["date_creation"]  = pd.to_datetime(df["date_creation"], errors="coerce")
        df["annee_creation"] = df["annee_creation"].astype("Int64")

    # mettre en cache sous CSV 
    df.to_csv(cache_file, index=False)
    print(f"{len(df)} établissements actifs sauvegardés dans {cache_file}")
    return df

# filtre Pandas 

def filtrer_par_ape(df: pd.DataFrame, codes_ape) -> pd.DataFrame:
    """codes_ape : str ou liste, ex. '62.01Z' ou ['62.01Z', '62.02A']"""
    if isinstance(codes_ape, str):
        codes_ape = [codes_ape]
    codes_norm = [c.replace(".", "").upper() for c in codes_ape]
    ape_norm   = df["ape"].str.replace(".", "", regex=False).str.upper()
    return df[ape_norm.isin(codes_norm)].copy()


def filtrer_par_annee(df: pd.DataFrame, annee_min: int = 2010) -> pd.DataFrame:
    """Garde les établissements créés à partir de annee_min."""
    return df[df["annee_creation"] >= annee_min].copy()

In [2]:
df_secteur = extraire_secteur(cp="67000")

print(f"\n{len(df_secteur)} établissements actifs dans le secteur.")
print(f"   Codes APE distincts : {df_secteur['ape'].nunique()}")
df_secteur.head()

Cache trouvé pour 67000 — chargement local (cache_sirene/cp_67000_actifs.csv)

40518 établissements actifs dans le secteur.
   Codes APE distincts : 494


,nom,siret,siren,ape,code_postal,ville,date_creation,annee_creation,tranche_effectif
0,GROUPE GOETZMANN,02592023200037,025920232,64.20Z,67000,STRASBOURG,2017-01-01,2017,NN
1,JACOT ET COMPAGNIE,03562038400020,035620384,45.20A,67000,STRASBOURG,2012-07-09,2012,NN
2,KIENTZ ET MLE KIENTZ,03786654800011,037866548,81.10Z,67000,STRASBOURG,1994-12-25,1994,NN
3,MONASTIRI CATHERINE,03786657100013,037866571,81.10Z,67000,STRASBOURG,1994-12-25,1994,01
4,BEMRICH JEAN,03786658900015,037866589,81.10Z,67000,STRASBOURG,1994-12-25,1994,NN


In [3]:
CODES_APE = "74.90B"   # str ou liste : ["62.01Z", "62.02A"]
ANNEE_MIN = 2010       # établissements créés à partir de cette année

df_filtre = filtrer_par_ape(df_secteur, CODES_APE)
df_filtre = filtrer_par_annee(df_filtre, annee_min=ANNEE_MIN)

print(f"{len(df_filtre)} entreprises — APE={CODES_APE!r}, créées ≥ {ANNEE_MIN}, actives")
df_filtre.sort_values("annee_creation", ascending=False)

165 entreprises — APE='74.90B', créées ≥ 2010, actives


,nom,siret,siren,ape,code_postal,ville,date_creation,annee_creation,tranche_effectif
40400,JOSE NKWENGA KUENGOUA,99538022700018,995380227,74.90B,67000,STRASBOURG,2026-01-04,2026,NN
39991,ALEXIS PIGNARD FRIEDRICH,99169386200021,991693862,74.90B,67000,STRASBOURG,2026-01-05,2026,NN
36644,ADRIEN BARTH,93893441100019,938934411,74.90B,67000,STRASBOURG,2025-01-03,2025,NN
36558,ADJI MAI MAMAN MALAM BOUCAR,93835318200018,938353182,74.90B,67000,STRASBOURG,2025-01-06,2025,NN
531,GUILLAUME BESSON,10048515000013,100485150,74.90B,67000,STRASBOURG,2025-10-02,2025,NN
...,...,...,...,...,...,...,...,...,...
16528,MICHAELA PREINER,52131274400019,521312744,74.90B,67000,STRASBOURG,2010-02-25,2010,NN
16256,CAROLINE LEVY,51913126200015,519131262,74.90B,67000,STRASBOURG,2010-01-05,2010,NN
2879,JEAN ATTIA,34285968300027,342859683,74.90B,67000,STRASBOURG,2010-01-01,2010,NN
17257,DA SOLAIRE SARL,52923149000010,529231490,74.90B,67000,STRASBOURG,2010-12-22,2010,NN


In [4]:
# charger la table NAF
df_naf = pd.read_csv(
    "int_courts_naf_rev_2.csv",
    header=None,
    usecols=[1, 2],
    names=["code_naf", "libelle_activite"],
    dtype=str,
).dropna(subset=["code_naf"])

# nettoyer df_secteur (évite les doublons si cellule relancée)
df_secteur = df_secteur.drop(
    columns=[c for c in df_secteur.columns if c in ("libelle_activite", "ape_norm", "code_naf_norm")],
)

# normaliser les clés (supprimer le point + majuscules) 
df_naf["code_naf_norm"] = df_naf["code_naf"].str.strip().str.replace(".", "", regex=False).str.upper()
df_secteur["ape_norm"]  = df_secteur["ape"].str.replace(".", "", regex=False).str.upper()

# jointure gauche
df_secteur = df_secteur.merge(
    df_naf[["code_naf_norm", "libelle_activite"]],
    left_on="ape_norm",
    right_on="code_naf_norm",
    how="left",
).drop(columns=["ape_norm", "code_naf_norm"])

In [5]:
df_secteur 

,nom,siret,siren,ape,code_postal,ville,date_creation,annee_creation,tranche_effectif,libelle_activite
0,GROUPE GOETZMANN,02592023200037,025920232,64.20Z,67000,STRASBOURG,2017-01-01,2017,NN,Activités des sociétés holding
1,JACOT ET COMPAGNIE,03562038400020,035620384,45.20A,67000,STRASBOURG,2012-07-09,2012,NN,Entretien et réparation de véhicules automobil...
2,KIENTZ ET MLE KIENTZ,03786654800011,037866548,81.10Z,67000,STRASBOURG,1994-12-25,1994,NN,Activités combinées de soutien lié aux bâtiments
3,MONASTIRI CATHERINE,03786657100013,037866571,81.10Z,67000,STRASBOURG,1994-12-25,1994,01,Activités combinées de soutien lié aux bâtiments
4,BEMRICH JEAN,03786658900015,037866589,81.10Z,67000,STRASBOURG,1994-12-25,1994,NN,Activités combinées de soutien lié aux bâtiments
...,...,...,...,...,...,...,...,...,...,...
40513,EUROPAISCHE UNION DER RECHTSPFLEGER (EUR),99995237700011,999952377,94.12Z,67000,STRASBOURG,2018-01-10,2018,NN,Activités des organisations professionnelles
40514,LEONOR LORENTZ,99996984300013,999969843,86.90D,67000,STRASBOURG,2026-02-01,2026,NN,Activités des infirmiers et des sages-femmes
40515,ORCHESTRE D'HARMONIE DE L'ACADEMIE DE STRASBOURG,99997458700019,999974587,90.01Z,67000,STRASBOURG,2025-07-25,2025,NN,Arts du spectacle vivant
40516,MARTIN GARBER,99997810900018,999978109,22.29B,67000,STRASBOURG,2026-01-12,2026,NN,Fabrication de produits de consommation couran...
